# Qualifire — Manual Smoke Test

Hands-on walkthrough of the most common features end-to-end. The notebook
uses **PySpark in `local[*]` mode** plus a **SQLite system table** so it
runs on a laptop with no external services.

> **Prerequisite:** `pyspark` and `qualifire` must be importable in the
> active kernel. If you're starting fresh, run
> [`pyspark_setup.ipynb`](pyspark_setup.ipynb) first — it covers venv
> creation, VS Code kernel selection, and the macOS Downloads-folder
> gotcha.

What you'll see, in order:

1. Bootstrap Spark + a partitioned `retail.sales_fact` table.
2. Append data **one partition at a time** so drift / forecast history can build up.
3. Run an **SLO + threshold** validation in a single `qf.validate(...)` call,
   demonstrating that multiple validations are grouped (not first-fail).
4. **Threshold** check via `validate_query()` — rendered SQL, NULL-safe metrics,
   and `partition_ts` stamping.
5. Inspect **Jinja2 templating** (`{{ ds }}`, `today`, `run_id`, ...).
6. Read raw rows from the **system table** (SQLite) — including the new
   `partition_ts` column.
7. Send a **notification** through a captured-in-memory channel and show the rendered message.
8. Render the **data-quality dashboard** (health report HTML) inline via `display(HTML(...))`.

## 1. Setup — Spark, warehouse, system table

In [ ]:
import os, sys

# Scrub anything Downloads-rooted: macOS Privacy blocks reading/exec'ing
# files under ~/Downloads for GUI-launched processes (Code/Code Helper),
# which corrupts both `import` discovery AND `spark-submit` invocation.
before = len(sys.path)
sys.path[:] = [p for p in sys.path if '/Downloads/' not in p]
os.environ.pop('PYTHONPATH', None)

# Drop SPARK_HOME if it points into Downloads — pyspark's bundled
# spark-submit (under .venv/.../site-packages/pyspark/bin) is what we want.
spark_home = os.environ.get('SPARK_HOME', '')
if '/Downloads/' in spark_home:
    print(f'unset SPARK_HOME (was {spark_home!r}; under ~/Downloads)')
    os.environ.pop('SPARK_HOME', None)

print(f'sys.path: kept {len(sys.path)}/{before} entries; PYTHONPATH unset')

In [ ]:
import os, re, glob, subprocess

# Spark 3.5 + Hadoop 3.3.4 require Java 8, 11, or 17. Java 18+ removes
# Subject.getSubject(AccessControlContext) which Hadoop calls on every
# SparkContext init -> "UnsupportedOperationException: getSubject is not
# supported". No JVM flag fixes this; we MUST pick a supported JDK.
#
# /usr/libexec/java_home is unreliable: `-v 1.8` falls back to the legacy
# JavaAppletPlugin (a JRE shim), and when the requested version isn't
# installed it returns the latest JDK instead of erroring. So we discover
# JDKs by globbing common locations + scraping the `-V` listing, then run
# `bin/java -version` against each to learn the *actual* major version.
# This installs nothing — only existing JDKs are considered.

ACCEPTABLE = (8, 11, 17)        # ordered by preference; matches .envrc=1.8

def _major(java_bin):
    try:
        out = subprocess.check_output(
            [java_bin, '-version'], stderr=subprocess.STDOUT, text=True,
        )
    except Exception:
        return None
    m = re.search(r'version "([^"]+)"', out)
    if not m:
        return None
    parts = m.group(1).split('.')
    return 8 if parts[:2] == ['1', '8'] else int(parts[0])

def _discover_homes():
    homes = []
    seen = set()
    def _add(h):
        if h and h not in seen and os.path.isdir(h):
            seen.add(h); homes.append(h)
    # Glob common system + user JDK locations.
    patterns = [
        '/Library/Java/JavaVirtualMachines/*/Contents/Home',
        os.path.expanduser('~/Library/Java/JavaVirtualMachines/*/Contents/Home'),
        '/opt/homebrew/Cellar/openjdk*/*/libexec/openjdk.jdk/Contents/Home',
        '/usr/local/Cellar/openjdk*/*/libexec/openjdk.jdk/Contents/Home',
        '/opt/homebrew/opt/openjdk*/libexec/openjdk.jdk/Contents/Home',
        os.path.expanduser('~/.sdkman/candidates/java/*'),
    ]
    for pat in patterns:
        for h in glob.glob(pat):
            _add(h)
    # Also scrape `java_home -V` to catch anything macOS knows about
    # that lives outside the patterns above.
    try:
        listing = subprocess.check_output(
            ['/usr/libexec/java_home', '-V'], text=True, stderr=subprocess.STDOUT,
        )
        for line in listing.splitlines():
            m = re.search(r'\s(\S+/Contents/Home)\s*$', line)
            if m:
                _add(m.group(1))
    except Exception:
        pass
    return homes

# Allow an explicit override (e.g. QF_JAVA_HOME=/path/to/jdk).
override = os.environ.get('QF_JAVA_HOME', '').strip()
if override:
    homes = [override]
else:
    homes = _discover_homes()

# Probe every candidate and keep only the acceptable ones, preferring 8 → 11 → 17.
inspected = []
acceptable = []
for h in homes:
    if 'JavaAppletPlugin' in h:
        inspected.append((h, None, 'rejected: JavaAppletPlugin (legacy JRE shim)'))
        continue
    java_bin = os.path.join(h, 'bin', 'java')
    if not os.path.isfile(java_bin):
        inspected.append((h, None, 'rejected: no bin/java'))
        continue
    mj = _major(java_bin)
    if mj is None:
        inspected.append((h, None, 'rejected: java -version unparseable'))
        continue
    if mj not in ACCEPTABLE:
        inspected.append((h, mj, f'rejected: Java {mj} (Spark 3.5 needs 8/11/17)'))
        continue
    inspected.append((h, mj, 'OK'))
    acceptable.append((h, mj))

# Stable sort: by ACCEPTABLE order (8 first), preserving discovery order within a tier.
acceptable.sort(key=lambda hm: ACCEPTABLE.index(hm[1]))

print('Java candidates inspected:')
for h, mj, why in inspected:
    print(f'  [{mj if mj is not None else "?":>2}] {why:50s} {h}')
print()

if not acceptable:
    raise RuntimeError(
        "No usable Java 8/11/17 JDK found. See list above. "
        "Set QF_JAVA_HOME=/path/to/jdk if your Java 8 lives somewhere unusual."
    )

java_home, mj = acceptable[0]
os.environ['JAVA_HOME'] = java_home
# Prepend so child processes (spark-submit shells out to `java`) pick THIS
# JDK, not whatever Java 24 might also be on the inherited PATH.
os.environ['PATH'] = f"{java_home}/bin:" + os.environ.get('PATH', '')
print(f'==> JAVA_HOME = {java_home}  (Java {mj})')
print(subprocess.check_output(
    [os.path.join(java_home, 'bin', 'java'), '-version'],
    stderr=subprocess.STDOUT, text=True,
).strip())

In [ ]:
import tempfile
from pathlib import Path

# This notebook assumes pyspark and qualifire are already importable in the
# active kernel. If not, run tests/manual/pyspark_setup.ipynb first.

# Make the JVM driver/workers spawn THIS interpreter, not whatever's first on PATH.
os.environ.setdefault('PYSPARK_PYTHON', sys.executable)
os.environ.setdefault('PYSPARK_DRIVER_PYTHON', sys.executable)

# Pin Spark driver IP so the JVM doesn't try to bind to an unresolved hostname.
os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')

WORK_DIR = Path(tempfile.mkdtemp(prefix='qualifire_manual_'))
WAREHOUSE = WORK_DIR / 'warehouse'
SYSTEM_DB = WORK_DIR / 'qualifire_history.sqlite'
REPORT_HTML = WORK_DIR / 'health_report.html'

print('Workdir   :', WORK_DIR)
print('Warehouse :', WAREHOUSE)
print('System DB :', SYSTEM_DB)

In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkContext

# Tear down any pre-existing JVM/gateway so PySpark spawns a *fresh* JVM
# that picks up the JAVA_HOME we set above. Without this step, a previous
# (failed) SparkContext init in the same kernel leaves a dead JVM behind,
# and getOrCreate() will reuse that JVM — typically pinned to whatever
# Java version was active at first launch.
if SparkContext._gateway is not None:
    try:
        SparkContext._gateway.shutdown()
    except Exception:
        pass
    SparkContext._gateway = None
    SparkContext._jvm = None
    SparkContext._active_spark_context = None
    print('shut down stale Py4J gateway / JVM')

# Hive metastore configuration — see notes:
# 1) `enableHiveSupport()` defaults to a Derby-backed metastore created at
#    `./metastore_db` in the kernel's CWD. Two kernels (or a re-run after a
#    crashed JVM) collide on Derby's exclusive file lock, producing
#    `Failed to start database 'metastore_db'` on every subsequent run
#    until the lock file is removed. Point each Spark session at a
#    metastore inside WORK_DIR so a) it's fresh per run, b) different
#    notebooks don't fight over a shared metastore in the project root.
# 2) `derby.system.home` is the parent directory Derby uses for its
#    internal `derby.log`. Put that in WORK_DIR too so the project root
#    stays clean.
metastore_url = f"jdbc:derby:;databaseName={WORK_DIR}/metastore_db;create=true"

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('qualifire-manual')
    .config('spark.sql.warehouse.dir', str(WAREHOUSE))
    .config('spark.sql.shuffle.partitions', '2')
    .config('javax.jdo.option.ConnectionURL', metastore_url)
    .config('spark.driver.extraJavaOptions', f'-Dderby.system.home={WORK_DIR}')
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

# Confirm the JVM we got is actually the Java 8/11/17 we wanted.
runtime = spark.sparkContext._jvm.System.getProperty('java.version')
print(f'JVM java.version = {runtime}')
assert runtime.startswith(('1.8.', '11.', '17.')), (
    f'Spark started with unsupported Java {runtime}. Restart the kernel '
    f'and re-run from the top so the cell-3 JAVA_HOME export takes effect.'
)

spark.sql('CREATE DATABASE IF NOT EXISTS retail')
spark.sql('DROP TABLE IF EXISTS retail.sales_fact')
spark

In [ ]:
from qualifire.api import Qualifire
from qualifire.backends.spark_backend import SparkBackend

qf = Qualifire(
    backend=SparkBackend(spark),
    system_table=str(SYSTEM_DB),
    system_table_backend='sqlite',
    owner='retail-data-quality',
    bu='retail',
)
qf

## 2. Build `retail.sales_fact` — one partition at a time

Real ETL pipelines land partitions one at a time, run validations against
each, and accumulate history in the system table. The cells below mirror
that flow with three building blocks:

- `seed_partition(date, …)` — write a single day's data.
- `validate_partition(date)` — run the day's standard checks.
- `process_partition(date, …)` — both, plus print the result summary.

Then we use those helpers two ways: a **loop** that processes the last 14
days as healthy baseline, and a **manual cell** that processes today's
partition with deliberate anomalies. After the loop you have real history
for the drift / trend / shape checks later in the notebook to compare
against.

Schema: `sale_id`, `product_id`, `amount`, `updated_at`, partitioned by `sale_date`.

In [ ]:
import random
from datetime import datetime, timedelta, date
from pyspark.sql import Row

from qualifire.core.exceptions import QualifireValidationError

random.seed(7)
TODAY = date.today()


# ---------------------------------------------------------------------------
# Per-partition helpers
# ---------------------------------------------------------------------------
# Real ETL pipelines land one partition at a time. The notebook mirrors that
# so drift / forecast / anomaly checks run with the same history-shape they'd
# see in production (each prior partition is its own qualifire run, persisted
# under that partition's `partition_ts`).
#
# Use the helpers in two ways:
#   - In a loop:  for d in date_range(...): process_partition(d, ...)
#   - Manually:   process_partition(date(2026, 4, 22), n_rows=1000, ...)


REGIONS = ['us', 'uk', 'jp', 'de']


def seed_partition(
    sale_date: date,
    n_rows: int = 1000,
    mean_amount: float = 50.0,
    null_pid_rate: float = 0.0,
    overwrite_table: bool = False,
):
    """Append a single day's partition to retail.sales_fact.

    `overwrite_table=True` recreates the table from scratch (use it for
    the very first partition to wipe any state from a previous run).
    """
    rows = []
    for i in range(n_rows):
        amt = max(1.0, random.gauss(mean_amount, mean_amount * 0.15))
        pid = None if random.random() < null_pid_rate else random.randint(1, 50)
        rows.append(Row(
            sale_id=f'{sale_date.isoformat()}-{i}',
            product_id=pid,
            region=random.choice(REGIONS),
            amount=float(round(amt, 2)),
            updated_at=datetime.combine(sale_date, datetime.min.time()) + timedelta(hours=12),
            sale_date=sale_date.isoformat(),
        ))
    df = spark.createDataFrame(rows)
    mode = 'overwrite' if overwrite_table else 'append'
    (
        df.write.mode(mode).partitionBy('sale_date')
        .format('parquet').saveAsTable('retail.sales_fact')
    )
    return n_rows


def validate_partition(sale_date: date, *, fail_quietly: bool = True):
    """Run the day's standard validations against `sale_date`.

    The threshold collector aggregates three metrics — row_count, avg_amount,
    null_pid_pct — and persists all of them as *collection rows* under this
    partition's `partition_ts`. Only `row_count` has a rule; the others get
    no validation rule but still land in the system table so downstream
    history-backed checks (drift / trend / shape later in this notebook)
    have data to compare against.
    """
    iso = sale_date.isoformat()
    try:
        result = qf.validate_query(
            name='retail_sales_fact',  # logical identity for history lookups
            partition_ts=f"'{iso}'",   # rendered to a literal date for partition-anchored reads
            query=f"""
                SELECT COUNT(*) AS row_count,
                       AVG(amount) AS avg_amount,
                       SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) * 100.0
                         / NULLIF(COUNT(*), 0) AS null_pid_pct
                FROM retail.sales_fact
                WHERE sale_date = '{iso}'
            """,
            validations=[
                Qualifire.threshold_check(
                    name='daily_metrics',
                    description='Daily row-count floor + collection-only metrics for history.',
                    aggregations={
                        'row_count':    'MAX(row_count)',
                        'avg_amount':   'MAX(avg_amount)',
                        'null_pid_pct': 'MAX(null_pid_pct)',
                    },
                    rules=[
                        {'metric': 'row_count', 'thresholds': {'error': {'min': 100}}},
                    ],
                ),
            ],
        )
    except QualifireValidationError as e:
        if not fail_quietly:
            raise
        result = e.result
    return result


def process_partition(
    sale_date: date,
    n_rows: int = 1000,
    mean_amount: float = 50.0,
    null_pid_rate: float = 0.0,
    overwrite_table: bool = False,
):
    """One full ETL+QA cycle for a single partition: write, then validate."""
    written = seed_partition(
        sale_date, n_rows=n_rows, mean_amount=mean_amount,
        null_pid_rate=null_pid_rate, overwrite_table=overwrite_table,
    )
    result = validate_partition(sale_date)
    summary = []
    for ds in result.datasets:
        for vr in ds.validation_results:
            summary.append((vr.severity.value, vr.validation_name, vr.message))
    return {'partition': sale_date.isoformat(), 'rows_written': written, 'results': summary}


def date_range(start: date, end: date) -> list[date]:
    """Inclusive list of dates between `start` and `end`."""
    days = (end - start).days
    return [start + timedelta(days=i) for i in range(days + 1)]

### 2a. Build up history — process the past N days, one partition at a time

Each `process_partition(d)` call seeds the day's data, runs the threshold
check, and persists results to the SQLite system table under
`partition_ts = d`. After the loop, drift / trend / shape have a real
history to compare against.

Note the very first call uses `overwrite_table=True` to drop any stale
table from an earlier kernel run.

In [ ]:
history_days = date_range(TODAY - timedelta(days=14), TODAY - timedelta(days=1))

for i, d in enumerate(history_days):
    out = process_partition(
        d,
        n_rows=1000,
        mean_amount=50.0,        # stable baseline
        null_pid_rate=0.0,       # no NULLs in healthy history
        overwrite_table=(i == 0),
    )
    severities = ' '.join(s for s, _, _ in out['results']) or '(no validations)'
    print(f"  {out['partition']}  rows={out['rows_written']:5d}  {severities}")

spark.sql(
    "SELECT sale_date, COUNT(*) AS rows, ROUND(AVG(amount), 2) AS avg_amt "
    "FROM retail.sales_fact GROUP BY sale_date ORDER BY sale_date"
).show(truncate=False)

### 2b. Process today's partition — anomalous on purpose

Today's partition has a 3× mean-amount spike and ~2% NULL `product_id`. The
history-backed checks below will compare today against the stable baseline
the loop above persisted, so drift / trend / shape will fire alerts.

To re-run a different day manually, swap `today_date` and re-execute this cell.

In [ ]:
today_date = TODAY  # set to date(2026, 4, 22) etc. to retry an earlier partition
# today_date = TODAY - timedelta(days=1)

today_summary = process_partition(
    today_date,
    n_rows=1000,
    mean_amount=150.0,    # 3x baseline → drift / trend / shape will see this as anomalous
    null_pid_rate=0.02,   # 2% NULLs → null-rate threshold will breach (later cells)
)
print(f"  {today_summary['partition']}  rows={today_summary['rows_written']}")
for severity, name, msg in today_summary['results']:
    print(f"    [{severity:>7}] {name:<40} — {msg[:140]}")

## 3. Two validations in one call — SLO + threshold (with `partition_ts`)

Demonstrates that `qf.validate(...)` runs **multiple validations against the
same dataset in a single call**: an SLO recency check and a row-count threshold,
both stamped with `partition_ts="'{{ ds }}'"` so the persisted rows carry the
partition identity used by historical lookups (anchor − k·step).

The engine runs *every* validation, persists *all* results, then raises
`QualifireValidationError` once if any landed at ERROR severity. We catch
that so the loop below can show both validations\' outcomes regardless.

`Qualifire.slo_check` accepts four `strategy=` values
(`max_column`, `delta_log`, `metadata`, `custom_sql`). For what each
strategy runs, the full configuration surface, and notification
routing, see:

- [`docs/validators/slo.md`](../../docs/validators/slo.md) — SLO check.
- [`docs/validators/threshold.md`](../../docs/validators/threshold.md) — threshold check.
- [`docs/notifications.md`](../../docs/notifications.md) — notify routing.

In [ ]:
from qualifire.core.exceptions import QualifireValidationError

# Two validations against the same dataset in a single call: an SLO check
# (recency) AND a threshold check (row-count floor). The engine runs both,
# persists their results, then raises QualifireValidationError if any landed
# at ERROR — wrapping in try/except keeps the result accessible so we can
# inspect every validation's outcome regardless of severity.
#
# `partition_ts="'{{ ds }}'"` is rendered through Jinja first to a literal
# date string, then injected by the aggregation collector as the partition
# stamp on every persisted row. Drift / forecast checks anchor their
# lookback at that timestamp via partition-anchored history reads
# (anchor − k·step).
try:
    multi_result = qf.validate(
        table='retail.sales_fact',
        partition_ts="'{{ ds }}'",
        validations=[
            Qualifire.slo_check(
                name='slo_freshness',
                column='updated_at', strategy='max_column',
                warning='PT24H', error='PT168H',  # forgiving: 1d/7d so demo data passes
            ),
            Qualifire.threshold_check(
                name='row_count_floor',
                aggregations={'rows': 'COUNT(*)'},
                rules=[{'metric': 'rows', 'thresholds': {'error': {'min': 1}}}],
            ),
        ],
    )
except QualifireValidationError as e:
    multi_result = e.result  # both validations still ran; surface both

print(f'Run id  : {multi_result.run_id}')
print('Validations (both ran in a single qf.validate call):')
for ds in multi_result.datasets:
    for vr in ds.validation_results:
        print(f'  [{vr.severity.value:>7}] {vr.validation_name:<30} — {vr.message}')

## 4. Threshold check after a SQL expression

`validate_query` materializes any SQL (Jinja-templated) and runs validations
against the result. Here we compute today's null-rate on `product_id` and
assert it stays under 0.5%. The seeded 2% null rate should trigger ERROR.

In [ ]:
try:
    threshold_result = qf.validate_query(
        name='retail.sales_fact',                      # logical identity = the table itself
        partition_ts="'{{ ds }}'",
        query="""
            SELECT * FROM retail.sales_fact WHERE sale_date = '{{ ds }}'
        """,
        validations=[
            # The query above returns RAW rows. The threshold validator
            # GROUPs BY the supplied `dimensions` and runs the
            # aggregations *inside* the validator — one validation per
            # (metric, dimension_combo). A bad region trips ERROR even
            # when the global aggregate would still pass.
            Qualifire.threshold_check(
                name='daily_minimums',
                description='Per-region row count + null-rate floor.',
                dimensions=['region'],
                aggregations={
                    'rows':           'COUNT(*)',
                    'null_pid_pct': (
                        "SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) "
                        "* 100.0 / NULLIF(COUNT(*), 0)"
                    ),
                },
                rules=[
                    {'metric': 'rows',         'thresholds': {'error':   {'min': 250}}},
                    {'metric': 'null_pid_pct', 'thresholds': {'error':   {'max': 1}}},
                ],
            ),
        ],
    )
except QualifireValidationError as e:
    threshold_result = e.result   # validations still ran; surface every per-region result

# One ValidationResult per (metric, dimension_value). The dimension JSON
# lands in `dimension_value`; reads via the dashboard / system table can
# slice by region.
for ds in threshold_result.datasets:
    for vr in ds.validation_results:
        dim = vr.dimension_value or '(no dim)'
        print(f'  [{vr.severity.value:>7}] {vr.validation_name:<30} dim={dim}')
        print(f'           {vr.message}')

## 4a. Drift (historical) — signed deviation + rate-of-change

`drift_check` reads the past N partitions for the same metric+dimension,
computes `deviation_pct` (signed: current vs. mean of past) and
`rate_of_change_pct` (signed: current vs. immediate-prior partition), then
applies signed `{min, max}` bounds. Today's `avg_amount` is 3× the baseline
mean so the check should land at ERROR.

Threshold form options for each measure:

- bare number `25` → symmetric absolute (`|value| > 25`) — back-compat.
- dict `{min: -10, max: 25}` → asymmetric signed bounds — the new form.

See [`docs/validators/drift.md`](../../docs/validators/drift.md) for
every measure (`deviation_pct`, `deviation_abs`, `z_score`,
`rate_of_change_pct`, `rate_of_change_abs`), cold-start handling, and
[`docs/validators/partition_anchoring.md`](../../docs/validators/partition_anchoring.md)
for the `partition_ts` / `step` contract drift relies on.

> **Reading this section's output**: cell 10 (the seeding loop)
> wrote 14 days of historical metric rows under their respective
> `partition_ts` values. `drift_check` reads those past rows from
> the system table — that's where `historical mean = 50.12` comes
> from, and why the check fires with ERROR despite this being your
> "first" `qf.validate(...)` call after the kernel started. **An
> ERROR severity here is the expected demo outcome** — today's
> `avg_amount` is 3× the baseline so the check is demonstrating
> its detection contract, not surfacing a real bug.


In [ ]:
try:
    drift_result = qf.validate_query(
        name='retail_sales_fact',  # SAME identity as validate_partition() so history matches
        partition_ts=f"'{today_date.isoformat()}'",
        query=f"""
            SELECT AVG(amount) AS avg_amount
            FROM retail.sales_fact
            WHERE sale_date = '{today_date.isoformat()}'
        """,
        validations=[
            Qualifire.drift_check(
                name='avg_amount_drift',
                rules=[{
                    'metric': 'avg_amount',
                    # past_values=3 + step=P1D ⇒ anchor − 1d, − 2d, − 3d
                    'compare': {'past_values': 3, 'step': 'P1D'},
                    'thresholds': {
                        # Asymmetric: a +25% spike fails error, a -50% drop fails error.
                        'warning': {'deviation_pct': {'min': -10, 'max': 10}},
                        'error':   {'deviation_pct': {'min': -50, 'max': 25}},
                    },
                }],
            ),
        ],
    )
except QualifireValidationError as e:
    drift_result = e.result

for ds in drift_result.datasets:
    for vr in ds.validation_results:
        d = vr.details or {}
        print(f"  [{vr.severity.value:>7}] {vr.validation_name:<30}")
        if 'deviation_pct' in d:
            print(f"     deviation_pct  : {d['deviation_pct']:+.1f}%")
            print(f"     rate_of_change : {d.get('rate_of_change_pct', 0):+.1f}% vs prev")
            print(f"     mean_past      : {d.get('mean_past', 0):.2f}, stddev: {d.get('stddev', 0):.2f}")
        print(f"     msg: {vr.message[:160]}")

## 4b. Trend (forecast) — Prophet on the daily history

`trend_check` fits a Prophet model to the past `history_count` partitions and
flags the current value if it falls outside the predicted confidence
intervals. With 14 days of stable history (mean=50) and today's spike (mean=150),
the forecast interval should be exceeded.

Reference: [`docs/validators/forecast.md`](../../docs/validators/forecast.md)
covers Prophet hyperparameters, prediction-band tuning, and cold-start
handling. Forecast shares the partition-anchored history contract with
drift — see
[`docs/validators/partition_anchoring.md`](../../docs/validators/partition_anchoring.md).

> **Reading this section's output**: same as drift above — the
> seeding loop wrote 14 stable-mean=50 partitions to the system
> table; Prophet fits its forecast on that history and the
> warning / error bands above are derived from those past values
> (band centred near 50, today's 150 is far outside). **An ERROR
> here is the expected demo outcome.**


In [ ]:
try:
    import prophet  # noqa: F401
    has_prophet = True
except ImportError:
    has_prophet = False

if not has_prophet:
    print('prophet not installed — skipping trend demo. `pip install prophet` to enable.')
else:
    try:
        trend_result = qf.validate_query(
            name='retail_sales_fact',
            partition_ts=f"'{today_date.isoformat()}'",
            query=f"""
                SELECT AVG(amount) AS avg_amount
                FROM retail.sales_fact
                WHERE sale_date = '{today_date.isoformat()}'
            """,
            validations=[
                Qualifire.trend_check(
                    name='avg_amount_forecast',
                    description='Daily avg_amount must stay inside Prophet confidence interval.',
                    metric='avg_amount',
                    history_count=14,
                    step='P1D',
                ),
            ],
        )
    except QualifireValidationError as e:
        trend_result = e.result

    for ds in trend_result.datasets:
        for vr in ds.validation_results:
            print(f"  [{vr.severity.value:>7}] {vr.validation_name:<30} — {vr.message[:200]}")

## 4c. Shape (anomaly) — Isolation Forest with SHAP explanations

`shape_check` fits an Isolation Forest on samples from past partitions and
scores today's sample for anomaly. With `explain=True` the validator returns
SHAP feature contributions — the columns most responsible for the anomaly.

Reference: [`docs/validators/shape.md`](../../docs/validators/shape.md)
for the encoding pipeline (numeric / boolean / categorical / complex),
tuning guide, and SHAP output shape. The complementary batch-level
two-sample classifier lives in
[`docs/validators/pattern.md`](../../docs/validators/pattern.md).

In [ ]:
try:
    import sklearn  # noqa: F401
    has_sklearn = True
except ImportError:
    has_sklearn = False

if not has_sklearn:
    print('scikit-learn not installed — skipping shape/pattern demos. `pip install scikit-learn shap`.')
else:
    iso = today_date.isoformat()
    try:
        shape_result = qf.validate(
            table='retail.sales_fact',
            partition_ts=f"'{iso}'",
            validations=[
                Qualifire.shape_check(
                    name='sales_shape',
                    description='Isolation Forest anomaly score on today vs. past 7 days of samples.',
                    n_records=500,
                    past_dates=7, step='P1D',
                    slice_column='sale_date',
                    slice_value=iso,
                    n_estimators=50, contamination=0.1,
                    explain=True, drop_complex=True,
                    exclude_columns=["sale_id", "updated_at"],   # slice_column auto-excluded by engine
                ),
            ],
        )
    except QualifireValidationError as e:
        shape_result = e.result

    for ds in shape_result.datasets:
        for vr in ds.validation_results:
            print(f"  [{vr.severity.value:>7}] {vr.validation_name:<30} — {vr.message[:200]}")
            shap = (vr.details or {}).get('top_contributing_features') or []
            if shap:
                print('     SHAP top features:')
                for feat in shap[:5]:
                    print(f"       {feat['feature']:<25} importance={feat['importance']:+.4f}")

## 4d. Pattern (Random Forest two-sample) — with SHAP

`pattern_check` trains a Random Forest classifier to distinguish current vs.
past samples; when AUC is high, the two distributions differ and the check
fires. SHAP shows which columns drive the separation.

Reference: [`docs/validators/pattern.md`](../../docs/validators/pattern.md)
for the leakage-control contract (you MUST list the partition / date /
ID columns in `exclude_columns` or AUC ≈ 1.0 every run), AUC
interpretation, and tuning.

In [ ]:
if not has_sklearn:
    print('scikit-learn not installed — skipping pattern demo.')
else:
    iso = today_date.isoformat()
    try:
        pattern_result = qf.validate(
            table='retail.sales_fact',
            partition_ts=f"'{iso}'",
            validations=[
                Qualifire.pattern_check(
                    name='sales_pattern',
                    description='RF two-sample classifier — flags distribution shift between today and history.',
                    n_records=500,
                    past_dates=3, step='P1D',
                    slice_column='sale_date',
                    slice_value=iso,
                    n_estimators=50, cv_folds=2,
                    explain=True, drop_complex=True,
                    exclude_columns=['sale_id', 'sale_date', 'updated_at'],
                    thresholds={'warning': {'auc': 0.65}, 'error': {'auc': 0.85}},
                ),
            ],
        )
    except QualifireValidationError as e:
        pattern_result = e.result

    for ds in pattern_result.datasets:
        for vr in ds.validation_results:
            print(f"  [{vr.severity.value:>7}] {vr.validation_name:<30} — {vr.message[:200]}")
            shap = (vr.details or {}).get('top_contributing_features') or []
            if shap:
                print('     SHAP top features driving the shift:')
                for feat in shap[:5]:
                    print(f"       {feat['feature']:<25} importance={feat['importance']:+.4f}")

## 4e. Profiling, custom-query, and metrics collectors

Three more collection types beyond aggregation+sample+recency. The profiler
auto-collects per-column stats (min/max/null/distinct/quantiles); the
custom-query collector lets you hand-write a SQL that produces metric rows;
the metrics collector provides a curated short-hand for common counters.

Reference: [`docs/profiling.md`](../../docs/profiling.md) for the
single-pass profiling engine, per-column stat selectors, and custom
analyzers.

In [ ]:
from qualifire.core.config import (
    CustomQueryCollectionConfig, DatasetConfig, MetricsCollectionConfig,
    ProfilingCollectionConfig, ThresholdRuleConfig, ThresholdLevels,
    ThresholdValidationConfig,
)

iso = today_date.isoformat()

# --- (a) Profiling: column-level stats (null share, distinct, min/max) ---
profile_validation = ThresholdValidationConfig(
    name='profile_amount_null_rate',
    description='Profiling collector — flag if amount is ever NULL.',
    collection=ProfilingCollectionConfig(
        columns=['amount'],
        stats=['count', 'null_count', 'min', 'max', 'mean', 'stddev'],
        filter=f"sale_date = '{iso}'",
    ),
    rules=[ThresholdRuleConfig(
        metric='amount.null_count',
        thresholds=ThresholdLevels(error={'max': 0}),
    )],
)

# --- (b) CustomQuery: full SQL with the metric in the SELECT list ---
custom_validation = ThresholdValidationConfig(
    name='top_product_revenue',
    description='Custom SQL — top product revenue must be a meaningful share of the day.',
    collection=CustomQueryCollectionConfig(
        sql=(f"SELECT MAX(rev) AS top_product_rev FROM "
             f"(SELECT product_id, SUM(amount) AS rev FROM retail.sales_fact "
             f" WHERE sale_date = '{iso}' GROUP BY product_id) t"),
    ),
    rules=[ThresholdRuleConfig(
        metric='top_product_rev',
        thresholds=ThresholdLevels(error={'min': 1}),
    )],
)

# --- (c) Metrics: built-in shorthand for common counters ---
# MetricsCollectionConfig.metrics is dict[str, str] — name → SQL expression.
# Each key produces a metric row in the system table under that name with
# the expression's value as `metric_value`. Compose your shorthand library
# of measures here and reference them by key in `rules`.
metrics_validation = ThresholdValidationConfig(
    name='basic_counts',
    description='Metrics collector — total + non-null counts.',
    collection=MetricsCollectionConfig(
        metrics={
            'row_count':       'COUNT(*)',
            'non_null_amount': 'COUNT(amount)',
            'distinct_pids':   'COUNT(DISTINCT product_id)',
        },
        filter=f"sale_date = '{iso}'",
    ),
    rules=[ThresholdRuleConfig(
        metric='row_count',
        thresholds=ThresholdLevels(error={'min': 100}),
    )],
)

ds_demo = DatasetConfig(
    table='retail.sales_fact',
    description='Demo dataset for non-aggregation collectors.',
    partition_ts=f"'{iso}'",
    validations=[profile_validation, custom_validation, metrics_validation],
)

from qualifire.core.config import QualifireConfig
from qualifire.core.context import QualifireContext
from qualifire.core.engine import QualifireEngine

cfg_demo = QualifireConfig(
    owner=qf.owner, bu=qf.bu, system_table=qf.system_table or '',
    datasets=[ds_demo],
)
engine_demo = QualifireEngine(
    backend=qf.backend, storage=qf._storage,
    context=QualifireContext(), config=cfg_demo,
)
try:
    other_result = engine_demo.run()
except QualifireValidationError as e:
    other_result = e.result

for ds in other_result.datasets:
    for vr in ds.validation_results:
        print(f"  [{vr.severity.value:>7}] {vr.validation_name:<30} — {vr.message[:200]}")

## 5. Inspect Jinja2 templating

`QualifireContext` is the Jinja sandbox used to expand template strings in
queries / filters / SQL bodies. Built-in vars: `today`, `yesterday`, `now`,
`ds`, `ds_nodash`, `run_id`. Filters: `date_add`, `date_format`. Anything
passed via `context=` is merged on top.

In [ ]:
from qualifire.core.context import QualifireContext

ctx = QualifireContext(extra_context={'env': 'demo', 'job': {'name': 'manual_test'}})
print('Built-in + extra variables:')
for k, v in ctx.get_variables().items():
    print(f'  {k:>10} = {v!r}')

tpl = (
    "SELECT * FROM retail.sales_fact "
    "WHERE sale_date BETWEEN '{{ ds | date_add(-7) }}' AND '{{ ds }}' "
    "  AND env = '{{ env }}'  -- run_id={{ run_id }}"
)
print()
print('Template     :', tpl)
print('Rendered     :', ctx.render(tpl))

## 6. Read raw rows from the system table

Every validation result lands in the system table. With `system_table_backend='sqlite'`
we can pop it open with `sqlite3` and inspect the schema + rows directly.

In [ ]:
import sqlite3
import pandas as pd

with sqlite3.connect(SYSTEM_DB) as conn:
    schema = pd.read_sql('PRAGMA table_info(qualifire_history)', conn)
    rows = pd.read_sql(
        """
        SELECT run_id, dataset_name, table_name, validation_name, validation_type,
               metric_name, metric_value, validation_status, validation_message,
               record_type, partition_ts, run_timestamp, dimension_value
          FROM qualifire_history
         ORDER BY run_timestamp DESC
        """,
        conn,
    )

print('--- qualifire_history schema ---')
print(schema[['name', 'type']].to_string(index=False))
print()
print(f'--- {len(rows)} rows persisted so far ---')
rows

## 7. Notifications — capture in memory and inspect

We register a tiny `CapturingNotifier` (subclass of `Notifier`) so we can
see the exact rendered message instead of hitting Slack/email. Routing the
threshold violation to it via `notify={'error': ['inbox']}`.

Reference: [`docs/notifications.md`](../../docs/notifications.md) for
the channel registry, dataset-level vs per-validation routing, success
notifications, and cross-dataset grouping.

In [ ]:
from qualifire.notification.base import Notifier
from qualifire.core.models import NotificationResult, Severity

class CapturingNotifier(Notifier):
    """Captures notification payloads in-memory so we can render them locally
    instead of hitting Slack/email."""
    def __init__(self):
        self.sent = []

    @property
    def channel_name(self) -> str:
        return 'inbox'

    def send(self, dataset_result, severity, owner, bu, datasets=None):
        message = self._format_message(dataset_result, severity, owner, bu, datasets)
        self.sent.append({
            'severity': severity.value,
            'dataset': dataset_result.dataset_name,
            'message': message,
        })
        return NotificationResult(
            channel=self.channel_name,
            severity=severity,
            status='sent',
            message='captured locally',
        )

inbox = CapturingNotifier()
qf.register_notifier('inbox', inbox)

# Each `qf.validate(...)` / `qf.validate_query(...)` call is a complete run
# with its own collection + validation + notification. To capture every
# failure across an end-to-end pipeline run in one notification cycle, run
# every validation you care about in a single call and route them all to
# the same channel via `notify={'error': ['inbox'], 'warning': ['inbox']}`.
#
# Earlier cells fired their own notifications (with no channels routed to
# inbox), so they don't show up here. This cell deliberately re-runs the
# day's checks — SLO + threshold + drift — under one umbrella, with all
# severities routed to inbox, to demonstrate multi-failure capture.
notify_routes = {'warning': ['inbox'], 'error': ['inbox']}
iso = today_date.isoformat()

try:
    notify_result = qf.validate_query(
        name='retail_notify_demo',
        partition_ts=f"'{iso}'",
        query=f"""
            SELECT
              COUNT(*) AS row_count,
              AVG(amount) AS avg_amount,
              SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) * 100.0 /
                NULLIF(COUNT(*), 0) AS null_pid_pct
            FROM retail.sales_fact
            WHERE sale_date = '{iso}'
        """,
        validations=[
            Qualifire.threshold_check(
                name='daily_minimums',
                # rules already name the source columns (row_count,
                # null_pid_pct) — the builder auto-derives the
                # metrics list from them, so no separate
                # ``aggregations=`` / ``metrics=`` block is needed.
                rules=[
                    {'metric': 'row_count',   'thresholds': {'error': {'min': 100}}},
                    {'metric': 'null_pid_pct','thresholds': {'error': {'max': 0.5}}},
                ],
                notify=notify_routes,
            ),
            Qualifire.drift_check(
                name='avg_amount_drift',
                # auto-derives metrics from the rules list — no
                # redundant ``aggregations={'avg_amount': 'MAX(avg_amount)'}``.
                rules=[{
                    'metric': 'avg_amount',
                    'compare': {'past_values': 3, 'step': 'P1D'},
                    'thresholds': {'error': {'deviation_pct': {'min': -50, 'max': 25}}},
                }],
                notify=notify_routes,
            ),
        ],
    )
except QualifireValidationError as e:
    notify_result = e.result   # validations all ran; surface their results

print(f'Captured {len(inbox.sent)} notification(s).')
for n in inbox.sent:
    print('=' * 70)
    print(f'severity = {n["severity"]}    dataset = {n["dataset"]}')
    print('-' * 70)
    print(n['message'])

## 8. Data-quality health dashboard

`qf.health_report(...)` aggregates everything in the system table over the
last `days` days; passing `output_path=` writes a self-contained HTML
dashboard. We render it inline.

Reference: [`docs/programmatic_api.md`](../../docs/programmatic_api.md)
→ "Dashboards" covers `qf.health_report`,
`qf.interactive_dashboard`, and the Plotly chart helpers under
`qualifire.reporting`. End-to-end notebook examples live in
[`tests/manual/dashboard_html.ipynb`](dashboard_html.ipynb) and
[`tests/manual/dashboard_charts.ipynb`](dashboard_charts.ipynb).

In [ ]:
report = qf.health_report(days=14, output_path=str(REPORT_HTML))
print('totals  :', report.total_checks, 'checks')
print('rates   : pass={:.1f}%  warn={:.1f}%  error={:.1f}%'.format(
    report.pass_rate, report.warning_rate, report.error_rate))
print('by_type :')
for row in report.by_check_type:
    print('   ', row)
print()
print(f'HTML dashboard written to: {REPORT_HTML}  ({REPORT_HTML.stat().st_size:,} bytes)')

In [ ]:
from IPython.display import HTML, display

# VS Code's notebook webview blocks `file://` srcs in <iframe> for security,
# which leaves the IFrame approach rendering blank. Read the file content
# and embed it directly with display(HTML(...)) — works in VS Code, classic
# Jupyter, JupyterLab, and nbviewer alike.
display(HTML(REPORT_HTML.read_text()))

## 8a. Interactive dashboard — drill down from datasets to a single validation

`qf.interactive_dashboard(...)` renders a self-contained HTML page with
Plotly charts loaded via CDN. Default view is "all datasets". Pick a
dataset to see its validations; pick a validation to see its per-partition
history (line chart with severity-coloured markers + threshold context).

Time range defaults to 90 days but is editable in the page. Dark/light
mode adapts to your OS.

In [ ]:
# Save a CDN-based copy to disk (small file; opens cleanly in any
# browser if you ever want to share the link), and embed an inline-Plotly
# copy directly in this notebook output for full interactivity without
# leaving the kernel. The inline path is required because VS Code's
# notebook webview sandboxes the iframe and blocks <script src="cdn...">.
INTERACTIVE_HTML = WORK_DIR / 'interactive_dashboard.html'
qf.interactive_dashboard(output_path=str(INTERACTIVE_HTML), days=90)
print(f'Interactive dashboard written to: {INTERACTIVE_HTML}')
print(f'Size (CDN form): {INTERACTIVE_HTML.stat().st_size:,} bytes')

# Inline render — Plotly JS bundled into the HTML (~3.5 MB).
#
# Two ways to embed:
#
# - ``as_iframe=False`` (default below) renders the dashboard
#   directly inside VS Code's own notebook output. Scripts run
#   in modern VS Code Jupyter extensions, and scrolling chains
#   to the next cell naturally because there's no extra iframe
#   boundary in between.
# - ``as_iframe=True`` wraps the dashboard in an
#   ``<iframe srcdoc="...">`` for older Jupyter setups that
#   strip inline ``<script>`` tags from ``display(HTML(...))``
#   output. Trade-off: scrolling cannot chain past the iframe
#   boundary (VS Code's webview is cross-origin, so the
#   iframe's ``window.parent.scrollBy`` is blocked) — you
#   need to move the mouse outside the iframe to scroll the
#   notebook.
#
# If the dropdowns appear empty with ``as_iframe=False``, your
# Jupyter setup is blocking inline scripts; flip to
# ``as_iframe=True`` (or open the saved HTML file in a browser).
display(HTML(qf.interactive_dashboard(
    days=90, inline_plotly_js=True, as_iframe=False,
)))

## 9. Cleanup (optional)

Stops the SparkSession but **keeps the workdir** so you can poke at the
warehouse, sqlite DB, and HTML report after the kernel shuts down. Uncomment
the `shutil.rmtree` line if you want a clean slate.

In [ ]:
import shutil
from pathlib import Path

spark.stop()
shutil.rmtree(WORK_DIR, ignore_errors=True)
print('artifacts removed from:', WORK_DIR)

# Also clean up Derby/Hive metastore turds that earlier *failed* runs may
# have left in the kernel's CWD before we wired the metastore URL into
# WORK_DIR. Leftover `metastore_db/` keeps an exclusive Derby file lock
# that breaks subsequent SparkSession.getOrCreate() with
# `Failed to start database 'metastore_db'`. Idempotent — no-op if absent.
cwd = Path.cwd()
for name in ('metastore_db', 'derby.log'):
    target = cwd / name
    if target.is_dir():
        shutil.rmtree(target, ignore_errors=True)
        print(f'  removed stray {target}/')
    elif target.is_file():
        target.unlink()
        print(f'  removed stray {target}')